# Notebook 6 — Comprensión avanzada de imágenes

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lección 6:** Advanced Images Understanding (monumentos, logotipos, objetos y Safe Search)

In [ ]:
import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision
client = vision.ImageAnnotatorClient()
print("Cliente listo.")


## 1. Detección de monumentos (`landmark_detection`)

`landmark_detection` reconoce estructuras naturales o construidas por el ser humano con relevancia cultural, histórica o geográfica — la Torre Eiffel, la Estatua de la Libertad, el Taj Mahal. Funciona incluso con el monumento parcialmente oculto, en ángulos inusuales o con poca luz.

In [ ]:
def detect_landmark(file_path):
    with open(file_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.landmark_detection(image=image)

    for annotation in response.landmark_annotations:
        print("Monumento:", annotation.description)
        print("Ubicación:", annotation.locations)
        print("Confianza:", annotation.score)

    return response.landmark_annotations


# detect_landmark("torre_eiffel.jpg")


## Ejercicio 1 — Landmarks en pandas, ordenados por confianza

Completa `landmarks_en_dataframe` para que procese una **lista** de rutas de imagen, llame a `detect_landmark` en cada una, y consolide todos los resultados en un único DataFrame de pandas con columnas `imagen`, `description` y `score`, ordenado de mayor a menor confianza.

In [ ]:
import pandas as pd

def landmarks_en_dataframe(rutas_imagenes):
    registros = []

    for ruta in rutas_imagenes:
        anotaciones = detect_landmark(ruta)
        # TODO: añade a "registros" un diccionario por cada anotación con las claves
        # "imagen", "description" y "score"
        ...

    df = pd.DataFrame(registros)
    # TODO: ordena el DataFrame por "score" de mayor a menor
    df = ...
    return df


# landmarks_en_dataframe(["torre_eiffel.jpg", "estatua_libertad.jpg"])


<details>
<summary><strong>Ver solución</strong></summary>

```python
import pandas as pd

def landmarks_en_dataframe(rutas_imagenes):
    registros = []

    for ruta in rutas_imagenes:
        anotaciones = detect_landmark(ruta)
        for anotacion in anotaciones:
            registros.append({
                "imagen": ruta,
                "description": anotacion.description,
                "score": anotacion.score,
            })

    df = pd.DataFrame(registros)
    df = df.sort_values("score", ascending=False)
    return df


landmarks_en_dataframe(["torre_eiffel.jpg", "estatua_libertad.jpg"])
```

</details>

## 2. Detección de logotipos (`logo_detection`)

`logo_detection` identifica marcas y organizaciones dentro de una imagen, incluso con el logotipo parcialmente oculto o distorsionado. En una prueba descrita en el material del curso, la API reconoció sin problema los logotipos de Apple, Nintendo, Best Buy y Nike, pero no logró identificar los de IBM, Circuit City y Playboy — la precisión global rondó el 90%.

In [ ]:
def detect_logos(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.logo_detection(image=image)
    logos = response.logo_annotations

    for logo in logos:
        print("Logo:", logo.description)
        print("Confianza:", logo.score)
        print("Cuadro delimitador:", logo.bounding_poly)

    return logos


# detect_logos("tienda_electronica.jpg")


## Ejercicio 2 — Estudio de falsos negativos

Completa `tasa_deteccion` para que, dada una lista de rutas de imágenes que contienen *un* logo conocido cada una y una lista paralela con el nombre esperado del logo, calcule el porcentaje de aciertos (imágenes en las que `logo_detection` detectó correctamente el logo esperado, comparando en minúsculas).

In [ ]:
def tasa_deteccion(rutas_imagenes, nombres_esperados):
    aciertos = 0

    for ruta, esperado in zip(rutas_imagenes, nombres_esperados):
        logos = detect_logos(ruta)
        # TODO: comprueba si "esperado" aparece entre los logos detectados (compara en minúsculas)
        detectado = ...
        if detectado:
            aciertos += 1

    return aciertos / len(rutas_imagenes) * 100


# tasa_deteccion(["gap.jpg", "nike.jpg", "audi.jpg"], ["gap", "nike", "audi"])


<details>
<summary><strong>Ver solución</strong></summary>

```python
def tasa_deteccion(rutas_imagenes, nombres_esperados):
    aciertos = 0

    for ruta, esperado in zip(rutas_imagenes, nombres_esperados):
        logos = detect_logos(ruta)
        detectado = any(logo.description.lower() == esperado.lower() for logo in logos)
        if detectado:
            aciertos += 1

    return aciertos / len(rutas_imagenes) * 100


tasa_deteccion(["gap.jpg", "nike.jpg", "audi.jpg"], ["gap", "nike", "audi"])
```

</details>

## 3. Detección de objetos (`object_localization`)

`object_localization` combina dos tareas: **clasificación** (qué es cada objeto) y **localización** (dónde está, con su cuadro delimitador). Internamente usa un modelo **SSD** (detector multibox de disparo único), una arquitectura de red neuronal convolucional (CNN) entrenada con millones de imágenes.

Fíjate en que `bounding_poly.normalized_vertices` devuelve coordenadas **normalizadas** entre cero y uno; para dibujarlas sobre la imagen real tendrás que multiplicarlas por el ancho y el alto (lo verás en el notebook 8, con Pillow).

In [ ]:
def detect_objects(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.object_localization(image=image)
    objects = response.localized_object_annotations

    for obj in objects:
        print(f"{obj.name} (confianza: {obj.score:.2f})")

    return objects


# detect_objects("salon.jpg")


## Ejercicio 3 — Efecto de la orientación en la confianza

Completa `comparar_angulos` para que, dado un diccionario `{"angulo": "ruta_imagen"}` de la misma escena fotografiada desde distintos ángulos, construya un DataFrame con la confianza media de todos los objetos detectados en cada ángulo, para comprobar si la orientación afecta a la fiabilidad de la detección.

In [ ]:
import pandas as pd

def comparar_angulos(imagenes_por_angulo):
    registros = []

    for angulo, ruta in imagenes_por_angulo.items():
        objetos = detect_objects(ruta)
        # TODO: calcula la confianza media de "objetos" (usa 0 si la lista está vacía)
        confianza_media = ...
        registros.append({"angulo": angulo, "confianza_media": confianza_media})

    return pd.DataFrame(registros)


# comparar_angulos({"frontal": "espejo_frontal.jpg", "lateral": "espejo_lateral.jpg"})


<details>
<summary><strong>Ver solución</strong></summary>

```python
import pandas as pd

def comparar_angulos(imagenes_por_angulo):
    registros = []

    for angulo, ruta in imagenes_por_angulo.items():
        objetos = detect_objects(ruta)
        scores = [obj.score for obj in objetos]
        confianza_media = sum(scores) / len(scores) if scores else 0
        registros.append({"angulo": angulo, "confianza_media": confianza_media})

    return pd.DataFrame(registros)


comparar_angulos({"frontal": "espejo_frontal.jpg", "lateral": "espejo_lateral.jpg"})
```

</details>

## 4. Detección de contenido explícito (`safe_search_detection`)

`safe_search_detection` estima la probabilidad de que una imagen contenga contenido inapropiado, en cinco categorías: **adult** (contenido sexual explícito), **spoof** (imágenes manipuladas o engañosas), **medical** (contenido médico o anatómico), **violence** (violencia) y **racy** (contenido sugerente).

Cada categoría devuelve un valor de la misma enumeración de verosimilitud que viste en la detección de rostros: `UNKNOWN`, `VERY_UNLIKELY`, `UNLIKELY`, `POSSIBLE`, `LIKELY` o `VERY_LIKELY`.

**Ten presente sus límites**: Safe Search puede producir falsos positivos y falsos negativos. Combínalo siempre con otras técnicas de moderación —revisión humana, reportes de la comunidad usuaria— en lugar de confiar en él como única barrera.

In [ ]:
LIKELIHOOD_NAMES = (
    "UNKNOWN",
    "VERY_UNLIKELY",
    "UNLIKELY",
    "POSSIBLE",
    "LIKELY",
    "VERY_LIKELY",
)

def check_safe_search(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.safe_search_detection(image=image)
    safe_search = response.safe_search_annotation

    return {
        "adult": LIKELIHOOD_NAMES[safe_search.adult],
        "spoof": LIKELIHOOD_NAMES[safe_search.spoof],
        "medical": LIKELIHOOD_NAMES[safe_search.medical],
        "violence": LIKELIHOOD_NAMES[safe_search.violence],
        "racy": LIKELIHOOD_NAMES[safe_search.racy],
    }


# print(check_safe_search("imagen.jpg"))


## Ejercicio 4 — Regla de moderación automática

Completa `requiere_revision` para que devuelva `True` si alguna de las cinco categorías de Safe Search alcanza `LIKELY` o `VERY_LIKELY` (índice cuatro o cinco en `LIKELIHOOD_NAMES`), y `False` en caso contrario.

In [ ]:
def requiere_revision(image_path):
    resultado = check_safe_search(image_path)

    # TODO: comprueba si algún valor de "resultado" es "LIKELY" o "VERY_LIKELY"
    revisar = ...
    return revisar


# print(requiere_revision("imagen.jpg"))


<details>
<summary><strong>Ver solución</strong></summary>

```python
def requiere_revision(image_path):
    resultado = check_safe_search(image_path)
    niveles_altos = {"LIKELY", "VERY_LIKELY"}
    revisar = any(valor in niveles_altos for valor in resultado.values())
    return revisar


print(requiere_revision("imagen.jpg"))
```

</details>

---

**Siguiente paso:** abre `07_Datos_Web.ipynb`.